In [3]:
# Install required libraries (safe)
!pip install pgmpy ipywidgets --quiet

# -------------------------------
# Imports
# -------------------------------
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
import ipywidgets as widgets
from IPython.display import display

# -------------------------------
# Dataset
# -------------------------------
data = pd.DataFrame([
    ['Old', 'High', 'High', 'Yes'],
    ['Old', 'High', 'Normal', 'Yes'],
    ['Middle', 'High', 'High', 'Yes'],
    ['Young', 'Normal', 'Normal', 'No'],
    ['Young', 'Normal', 'High', 'No'],
    ['Middle', 'Normal', 'High', 'Yes'],
    ['Old', 'Normal', 'High', 'Yes'],
    ['Young', 'High', 'Normal', 'No'],
    ['Middle', 'High', 'Normal', 'Yes'],
    ['Old', 'High', 'High', 'Yes'],
    ['Young', 'Normal', 'Normal', 'No'],
    ['Middle', 'Normal', 'Normal', 'No']
], columns=['Age', 'Cholesterol', 'BP', 'HeartDisease'])

# -------------------------------
# Model (UPDATED - no error)
# -------------------------------
model = DiscreteBayesianNetwork([
    ('Age', 'HeartDisease'),
    ('Cholesterol', 'HeartDisease'),
    ('BP', 'HeartDisease')
])

# Train model
model.fit(data, estimator=MaximumLikelihoodEstimator)

# Inference
inference = VariableElimination(model)

# -------------------------------
# UI Components
# -------------------------------
age = widgets.Dropdown(options=['Young', 'Middle', 'Old'], description='Age:')
chol = widgets.Dropdown(options=['Normal', 'High'], description='Chol:')
bp = widgets.Dropdown(options=['Normal', 'High'], description='BP:')

button = widgets.Button(description="Predict", button_style='success')
output = widgets.Output()

# -------------------------------
# Prediction Function
# -------------------------------
def predict(b):
    with output:
        output.clear_output()

        result = inference.query(
            variables=['HeartDisease'],
            evidence={
                'Age': age.value,
                'Cholesterol': chol.value,
                'BP': bp.value
            }
        )

        # Correct indexing
        prob_yes = round(result.values[1], 2)
        prob_no = round(result.values[0], 2)

        print("🔍 Prediction Result:")
        print(f"P(HeartDisease=Yes): {prob_yes}")
        print(f"P(HeartDisease=No): {prob_no}")

        if prob_yes > prob_no:
            print("⚠️ High Risk")
        else:
            print("✅ Low Risk")

# Button action
button.on_click(predict)

# -------------------------------
# Display UI
# -------------------------------
display(age, chol, bp, button, output)

Dropdown(description='Age:', options=('Young', 'Middle', 'Old'), value='Young')

Dropdown(description='Chol:', options=('Normal', 'High'), value='Normal')

Dropdown(description='BP:', options=('Normal', 'High'), value='Normal')

Button(button_style='success', description='Predict', style=ButtonStyle())

Output()